In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 04. Attention MHA GQA | 注意力机制与键值缓存 (MHA / GQA / MQA)

> 读取巨量的 KV Cache 会面临严重的**显存容量瓶颈**和**内存带宽瓶颈 (Memory-bound)**，导致推理极慢。

> * **MHA (Multi-Head Attention)**
> * **MQA (Multi-Query Attention)**
> * **GQA (Grouped-Query Attention)**
> *  **MLA(Multi-head Latent Attention)**

下面是一个GQA的实现

In [2]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    将 KV 头复制 n_rep 次，以匹配 Query 头的数量 (GQA/MQA 需要)
    """
    batch, num_kv_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_kv_heads * n_rep, slen, head_dim)

class GroupedQueryAttention(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int, num_kv_heads: int = None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads
        
        self.num_queries_per_kv = self.num_heads // self.num_kv_heads
        self.head_dim = hidden_dim // num_heads
        
        # 定义投影矩阵
        self.q_proj = nn.Linear(hidden_dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, hidden_dim, bias=False)

    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: torch.Tensor = None, 
        kv_cache: tuple[torch.Tensor, torch.Tensor] = None
    ):
        batch_size, seq_len, _ = x.shape
        
        # 1. 线性投影
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        
        # ==========================================
        # TODO 1: Reshape xq, xk, xv 以适配多头注意力计算
        # ==========================================
        xq = xq.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        xk = xk.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        xv = xv.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        # xq = ???
        # xk = ???
        # xv = ???
        
        
        # ==========================================
        # TODO 2: 处理 KV Cache
        # ==========================================
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            xk = torch.cat([k_cache, xk], dim=2)
            xv = torch.cat([v_cache, xv], dim=2)
            
        new_kv_cache = (xk, xv)
        
        # 通过 repeat_kv 把 GQA 的 KV 头数扩充到和 Query 数量一致
        xk = repeat_kv(xk, self.num_queries_per_kv)
        xv = repeat_kv(xv, self.num_queries_per_kv)
        
        # ==========================================
        # TODO 3: 计算注意力分数 (Scaled Dot-Product)
        # ==========================================
        scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            scores = scores + attention_mask
            
        probs = torch.nn.functional.softmax(scores, dim=-1)
        output = torch.matmul(probs, xv)
        
        # ==========================================
        # TODO 4: 恢复形状并输出
        # [B, H, S, D] -> [B, S, H*D]
        # ==========================================
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.o_proj(output), new_kv_cache
        pass


**进阶思考：GQA 的延迟扩充 (Lazy Expansion)**

- **为什么不直接缓存扩充后的 KV？** 如果在缓存时就用 `repeat_kv` 扩充，显存占用会和 MHA 一样大，失去了 GQA 的优势。
- **正确做法：** 只缓存原始的 `num_kv_heads` 个头，在每次前向传播时临时扩充。虽然增加了计算量，但由于注意力计算是 Memory-bound（受限于显存带宽而非计算速度），这个开销可以忽略。
- **工业实践：** vLLM、TensorRT-LLM 等推理框架都采用这种延迟扩充策略，在 70B 模型上可以节省数十 GB 的 KV Cache 显存。
